# link_setup_time module usage

This notebook shows the reusable LST workflow: build a target topology series, sweep one or more setup times, and write summary outputs. Each code cell is self-contained enough to run by itself.

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\\paper11\\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.config.viewer_config import G60_CONFIG
from src.link_setup_time.module import (
    build_region_internal_target_series,
    group_state_sequence_from_group_data,
    sweep_link_setup_times,
    write_sweep_outputs,
)
from src.satellite_topology_viewer.module.region_groups import load_or_build_group_data
from src.topology_workflow.module import build_single_motif_edge_table

MOTIF = {
    "name": "motif_2x2_CB",
    "w": 2,
    "h": 2,
    "offsets": {"A": [1, 0], "B": [1, -1], "C": [1, 1], "D": [2, 0]},
    "support": [[0, 0, "C"], [0, 1, "B"]],
}
STEPS = list(range(0, 301, 1))
XML_FILE = Path(r"E:\\paper11\\data\\basic_file\\G60\\satellitesposition\\station_visible_satellites_20250106.xml")
GROUP_CACHE_DIR = Path(r"E:\\paper11\\data\\satnet_experiments\\caches\\G60\\group_data_cache")
OUT_DIR = Path(r"E:\\paper11\\data\\satnet_experiments\\runs\\paper1\\G60\\link_setup_time\\notebook_example_0_300")

base_edge_table = build_single_motif_edge_table(
    topology_raw={
        "kind": "single_motif",
        "motif": MOTIF,
        "tiling": {"allow_vertical_overlap": True, "allow_clipped_right": True},
        "add_intra_ring": True,
    },
    config=G60_CONFIG,
)
group_data = load_or_build_group_data(
    xml_file=XML_FILE,
    group_cache_dir=GROUP_CACHE_DIR,
    steps=STEPS,
    station_groups=G60_CONFIG.station_groups,
    total_sats=G60_CONFIG.total_sats,
    constellation_name=G60_CONFIG.name,
    stride=1,
    enabled=True,
    force=False,
)
state_sequence = group_state_sequence_from_group_data(
    group_data=group_data,
    steps=STEPS,
    constrained_groups=(2, 3),
)
target_series = build_region_internal_target_series(
    config=G60_CONFIG,
    base_edge_table=base_edge_table,
    state_sequence=state_sequence,
    forced_option=0,
    wrap_planes=False,
)
result = sweep_link_setup_times(
    target_series=target_series,
    steps=STEPS,
    lst_values=(10, 20, 30),
    chunk_size=1000,
)
result = write_sweep_outputs(out_dir=OUT_DIR, result=result)
[(row.lst_s, round(row.mean_building_edges, 3), row.max_building_edges) for row in result.rows]